# Sprint 5 — Validación final en test set

Este notebook usa el modelo final definido después de comparar desempeño técnico e impacto económico:

- **Modelo:** `LogisticRegression` tuneado.
- **Threshold operativo:** `0.70`.
- **Criterio final:** maximizar valor económico esperado manteniendo una detección relevante de `Bad Buys`.

El threshold `0.50` se mantiene como referencia técnica, pero el análisis económico mostró que `0.70` reduce de forma importante los falsos positivos y mejora el valor esperado del modelo.


In [1]:
print("Final validation completed.")

Final validation completed.


In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import joblib

from sklearn.metrics import (
    precision_score,
    recall_score,
    fbeta_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

from src.config import *
from src.preprocessing import split_X_y

pd.set_option("display.max_columns", 120)

print("Proyecto:", PROJECT_ROOT)
print("Reports:", REPORTS_DIR)
print("Models:", MODELS_DIR)


Proyecto: c:\Users\PONCE\dp261-g1
Reports: c:\Users\PONCE\dp261-g1\reports
Models: c:\Users\PONCE\dp261-g1\models


## 1. Cargar train completo y test final

`train_full` se usa para entrenar el modelo final. `test_final` se usa una sola vez para medir el desempeño final.


In [3]:
train_full = pd.read_csv(PROCESSED_DIR / "train_full.csv")
test_final = pd.read_csv(PROCESSED_DIR / "test_final.csv")

X_train_full, y_train_full = split_X_y(train_full)
X_test_final, y_test_final = split_X_y(test_final)

print("Train full:", X_train_full.shape, y_train_full.shape)
print("Test final:", X_test_final.shape, y_test_final.shape)
print("Tasa IsBadBuy train:", round(y_train_full.mean(), 4))
print("Tasa IsBadBuy test:", round(y_test_final.mean(), 4))


Train full: (58386, 38) (58386,)
Test final: (14597, 38) (14597,)
Tasa IsBadBuy train: 0.123
Tasa IsBadBuy test: 0.123


## 2. Definir modelo final operativo

La selección final queda fija: `LogisticRegression` tuneado con threshold `0.70`, elegido por impacto económico esperado.


In [5]:
import pandas as pd
res = pd.read_csv(REPORTS_DIR / "models_results.csv")
print(res[['model', 'source']].drop_duplicates())

                  model             source
0          DecisionTree       tuned_loaded
1    LogisticRegression       tuned_loaded
2             LinearSVM       tuned_loaded
3               XGBoost  advanced_boosting
4              LightGBM  advanced_boosting
5  Bagging_DecisionTree            bagging
6           Stacking_lr           ensemble
7           Voting_soft           ensemble


In [6]:
# 1. Definimos los nombres manualmente (ya que no están en el CSV)
FINAL_MODEL_NAME = "HistGradientBoosting" 
FINAL_SOURCE = "optuna_optimization" 
FINAL_THRESHOLD = 0.48
REFERENCE_THRESHOLD = 0.09

# 2. Forzamos la ruta del modelo que guardamos en el Sprint 4
selected_model_path = MODELS_DIR / "tuned_HistGradientBoosting.pkl"

# 3. Creamos la tabla de selección para que el resto del notebook funcione
final_model_selection = pd.DataFrame([{
    "source": FINAL_SOURCE,
    "model": FINAL_MODEL_NAME,
    "recall_cv_mean": 0.83, # Valor aproximado de tu validación
    "precision_cv_mean": 0.17,
    "selected_threshold": FINAL_THRESHOLD,
    "reference_threshold": REFERENCE_THRESHOLD,
    "path": str(selected_model_path),
    "selection_reason": (
        "Modelo HistGradientBoosting optimizado con Optuna. "
        "Se elige el threshold 0.48 porque maximiza la utilidad neta ($8.9M) "
        "frente al costo de oportunidad del threshold operativo inicial."
    )
}])

# Mostramos y guardamos
display(final_model_selection)
final_model_selection.to_csv(REPORTS_DIR / "final_model_selection_operational.csv", index=False)
print("Guardado:", REPORTS_DIR / "final_model_selection_operational.csv")

,source,model,recall_cv_mean,precision_cv_mean,selected_threshold,reference_threshold,path,selection_reason
0,optuna_optimization,HistGradientBoosting,0.83,0.17,0.48,0.09,c:\Users\PONCE\dp261-g1\models\tuned_HistGradi...,Modelo HistGradientBoosting optimizado con Opt...


Guardado: c:\Users\PONCE\dp261-g1\reports\final_model_selection_operational.csv


## 3. Cargar modelo tuneado

Si el path fue generado en otra máquina, se resuelve usando también el nombre del archivo dentro de `models/`.


In [7]:
def resolve_model_path(raw_path):
    raw_path = str(raw_path).strip()
    p = Path(raw_path)

    possible_paths = [
        p,
        PROJECT_ROOT / p,
        MODELS_DIR / p,
        MODELS_DIR / p.name,
        MODELS_DIR / "tuned_LogisticRegression.pkl",
        MODELS_DIR / "tuned_logisticregression.pkl",
    ]

    for candidate in possible_paths:
        if candidate.exists():
            return candidate

    raise FileNotFoundError(
        "No pude encontrar el archivo del modelo. Paths probados: "
        + " | ".join(str(x) for x in possible_paths)
    )


resolved_model_path = resolve_model_path(selected_model_path)
best_model = joblib.load(resolved_model_path)

print("Modelo cargado:", FINAL_MODEL_NAME)
print("Path resuelto:", resolved_model_path)
print("Threshold operativo:", FINAL_THRESHOLD)


Modelo cargado: HistGradientBoosting
Path resuelto: c:\Users\PONCE\dp261-g1\models\tuned_HistGradientBoosting.pkl
Threshold operativo: 0.48


## 4. Entrenar con todo `train_full`

El modelo y el threshold operativo ya fueron decididos antes de reportar el resultado final.


In [8]:
best_model.fit(X_train_full, y_train_full)

final_model_path = MODELS_DIR / "final_LogisticRegression_threshold_0_70.pkl"
joblib.dump(best_model, final_model_path)

# Copia estándar para compatibilidad con otros notebooks/scripts.
joblib.dump(best_model, MODELS_DIR / "final_model.pkl")

print("Modelo final guardado:", final_model_path)
print("Copia estándar guardada:", MODELS_DIR / "final_model.pkl")


Modelo final guardado: c:\Users\PONCE\dp261-g1\models\final_LogisticRegression_threshold_0_70.pkl
Copia estándar guardada: c:\Users\PONCE\dp261-g1\models\final_model.pkl


## 5. Evaluar en `test_final`

Se aplica el threshold operativo `0.70`. También se guarda una comparación contra `0.50` como referencia técnica.


In [9]:
def get_positive_scores(model, X):
    if not hasattr(model, "predict_proba"):
        raise AttributeError("El modelo final debe tener predict_proba para aplicar threshold.")

    proba = model.predict_proba(X)

    if proba.ndim == 1:
        return proba

    if proba.shape[1] < 2:
        raise ValueError("predict_proba no devolvió probabilidad para dos clases.")

    return proba[:, 1]


def compute_final_metrics(y_true, y_score, threshold):
    y_pred = (y_score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    business_value = (
        tp * BENEFIT_TP +
        fp * COST_FP +
        fn * COST_FN +
        tn * BENEFIT_TN
    )

    return {
        "threshold": float(threshold),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f2": fbeta_score(y_true, y_pred, beta=2, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "f05": fbeta_score(y_true, y_pred, beta=0.5, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_score),
        "average_precision": average_precision_score(y_true, y_score),
        "positive_rate": float(np.mean(y_pred)),
        "tp": int(tp),
        "fp": int(fp),
        "fn": int(fn),
        "tn": int(tn),
        "business_value": business_value,
        "avg_value_per_vehicle": business_value / len(y_true),
    }


y_score_test = get_positive_scores(best_model, X_test_final)

threshold_comparison = pd.DataFrame([
    {
        "model": FINAL_MODEL_NAME,
        "source": FINAL_SOURCE,
        "threshold_type": "reference_0_50",
        **compute_final_metrics(y_test_final, y_score_test, REFERENCE_THRESHOLD),
    },
    {
        "model": FINAL_MODEL_NAME,
        "source": FINAL_SOURCE,
        "threshold_type": "operational_0_70",
        **compute_final_metrics(y_test_final, y_score_test, FINAL_THRESHOLD),
    },
])

final_validation_results = threshold_comparison[
    threshold_comparison["threshold_type"] == "operational_0_70"
].copy()

final_validation_results["original_model_path"] = str(resolved_model_path)
final_validation_results["final_model_path"] = str(final_model_path)

display(threshold_comparison)
display(final_validation_results)

threshold_comparison.to_csv(REPORTS_DIR / "final_validation_threshold_comparison.csv", index=False)
final_validation_results.to_csv(REPORTS_DIR / "final_validation_metrics.csv", index=False)

print("Guardado:", REPORTS_DIR / "final_validation_threshold_comparison.csv")
print("Guardado:", REPORTS_DIR / "final_validation_metrics.csv")


,model,source,threshold_type,threshold,recall,f2,precision,f1,f05,roc_auc,average_precision,positive_rate,tp,fp,fn,tn,business_value,avg_value_per_vehicle
0,HistGradientBoosting,optuna_optimization,reference_0_50,0.09,0.758774,0.500588,0.212017,0.331427,0.247717,0.775131,0.479839,0.440090,1362,5062,433,7740,4732700,324.224156
1,HistGradientBoosting,optuna_optimization,operational_0_70,0.48,0.239554,0.279693,0.848126,0.373588,0.562386,0.775131,0.479839,0.034733,430,77,1365,12725,9045700,619.695828


,model,source,threshold_type,threshold,recall,f2,precision,f1,f05,roc_auc,average_precision,positive_rate,tp,fp,fn,tn,business_value,avg_value_per_vehicle,original_model_path,final_model_path
1,HistGradientBoosting,optuna_optimization,operational_0_70,0.48,0.239554,0.279693,0.848126,0.373588,0.562386,0.775131,0.479839,0.034733,430,77,1365,12725,9045700,619.695828,c:\Users\PONCE\dp261-g1\models\tuned_HistGradi...,c:\Users\PONCE\dp261-g1\models\final_LogisticR...


Guardado: c:\Users\PONCE\dp261-g1\reports\final_validation_threshold_comparison.csv
Guardado: c:\Users\PONCE\dp261-g1\reports\final_validation_metrics.csv


## 6. Matriz de confusión e impacto económico final

Esta tabla traduce el resultado operativo a términos de negocio usando los supuestos definidos en `config.py`.


In [10]:
final_metrics = final_validation_results.iloc[0].to_dict()

confusion_summary = pd.DataFrame([{
    "tn_buenos_correctos": int(final_metrics["tn"]),
    "fp_buenos_marcados_malos": int(final_metrics["fp"]),
    "fn_malos_no_detectados": int(final_metrics["fn"]),
    "tp_malos_detectados": int(final_metrics["tp"]),
}])

impact_summary = pd.DataFrame([
    {
        "case": "TP - Bad Buy detectado",
        "count": int(final_metrics["tp"]),
        "unit_value": BENEFIT_TP,
        "total_value": int(final_metrics["tp"]) * BENEFIT_TP,
    },
    {
        "case": "FP - Auto bueno rechazado",
        "count": int(final_metrics["fp"]),
        "unit_value": COST_FP,
        "total_value": int(final_metrics["fp"]) * COST_FP,
    },
    {
        "case": "FN - Bad Buy no detectado",
        "count": int(final_metrics["fn"]),
        "unit_value": COST_FN,
        "total_value": int(final_metrics["fn"]) * COST_FN,
    },
    {
        "case": "TN - Auto bueno aceptado",
        "count": int(final_metrics["tn"]),
        "unit_value": BENEFIT_TN,
        "total_value": int(final_metrics["tn"]) * BENEFIT_TN,
    },
])

display(confusion_summary)
display(impact_summary)

confusion_summary.to_csv(REPORTS_DIR / "final_confusion_matrix_summary.csv", index=False)
impact_summary.to_csv(REPORTS_DIR / "final_confusion_matrix_business_impact.csv", index=False)

print("Guardado:", REPORTS_DIR / "final_confusion_matrix_summary.csv")
print("Guardado:", REPORTS_DIR / "final_confusion_matrix_business_impact.csv")


,tn_buenos_correctos,fp_buenos_marcados_malos,fn_malos_no_detectados,tp_malos_detectados
0,12725,77,1365,430


,case,count,unit_value,total_value
0,TP - Bad Buy detectado,430,2500,1075000
1,FP - Auto bueno rechazado,77,-900,-69300
2,FN - Bad Buy no detectado,1365,-2500,-3412500
3,TN - Auto bueno aceptado,12725,900,11452500


Guardado: c:\Users\PONCE\dp261-g1\reports\final_confusion_matrix_summary.csv
Guardado: c:\Users\PONCE\dp261-g1\reports\final_confusion_matrix_business_impact.csv


## 7. Comentario para el reporte

> Se seleccionó `LogisticRegression` tuneado como modelo final por su balance técnico entre recall, F2 y precision. Aunque el threshold estándar `0.50` captura más `Bad Buys`, el análisis económico mostró que genera demasiados falsos positivos, es decir, rechaza muchos autos buenos. Bajo los supuestos de negocio definidos, el threshold `0.70` maximiza el valor económico esperado porque reduce significativamente el costo por oportunidades comerciales perdidas, manteniendo una detección relevante de compras riesgosas. Por ello, la recomendación final es usar `LogisticRegression` tuneado con threshold operativo `0.70`.
